# Prompt Ablation Study - 3. Spanish (es) BT QA

This notebook runs BT QA for Spanish (es) for ALL 3 prompt strategies.

**Prerequisites:** Run `1_source_qa.ipynb` first.

## Environment Setup

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CACHE_DIR = '/content/drive/MyDrive/AskQE_Models_Cache'
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = DRIVE_CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = os.path.join(DRIVE_CACHE_DIR, 'transformers')

In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'torch', 'accelerate'], check=True)
print('Dependencies installed!')

In [ ]:
if IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone', 'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git', PROJECT_ROOT], check=True)
elif IN_COLAB:
    PROJECT_ROOT = '/content/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone', 'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git', PROJECT_ROOT], check=True)
else:
    PROJECT_ROOT = os.getcwd()
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
print(f'Loading {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
del model, tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('Model cached')

## Path Configuration

In [ ]:
ABLATION_DIR = f'{PROJECT_ROOT}/results Qwen3B baseline/biomqm/prompt-ablation'
CODE_DIR = f'{ABLATION_DIR}/code'
QG_PATH = f'{PROJECT_ROOT}/results Qwen3B baseline/biomqm/baseline/QG/qwen-3b.jsonl'
STRATEGIES = ['P1-fewshot', 'P2-cot', 'P3-concise']
LANG = 'es'
for strategy in STRATEGIES:
    os.makedirs(f'{ABLATION_DIR}/QA/{strategy}', exist_ok=True)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
print(f'Language: {LANG}')

## BT QA - Spanish (es)

In [ ]:
for strategy in STRATEGIES:
    output_file = f'{ABLATION_DIR}/QA/{strategy}/bt-{LANG}-{strategy}.jsonl'
    cmd = [sys.executable, '-u', f'{CODE_DIR}/qa_ablation.py',
           '--strategy', strategy, '--mode', 'bt', '--lang', LANG,
           '--qg_input_path', QG_PATH, '--output_path', output_file]
    print(f'Running BT QA {LANG} with {strategy}...')
    subprocess.run(cmd, check=True)
    print(f'Done {strategy}!\n')

## Git Push

In [ ]:
os.chdir(PROJECT_ROOT)
subprocess.run(['git', 'config', '--global', 'user.email', 'simone@example.com'])
subprocess.run(['git', 'config', '--global', 'user.name', 'Simone'])
subprocess.run(['git', 'add', '-A'])
subprocess.run(['git', 'commit', '-m', f'Add prompt ablation BT QA for {LANG}'])
subprocess.run(['git', 'push', 'origin', 'main'])
print('Push complete!')